# Objectif

Identifier les véhicules qui émettent le plus de CO2 est important pour identifier les caractéristiques techniques qui jouent un rôle dans la pollution. Prédire à l’avance cette pollution permet de prévenir dans le cas de l’apparition de nouveaux types de véhicules (nouvelles séries de voitures par exemple).

# Analyse et Nettoyage du dataset UE

Seuls les véhicules des années 2022-2023 et de la France ont été exportés à partir de https://www.eea.europa.eu/data-and-maps/data/co2-cars-emission-20

# <font color='#3585CD'>Importation des librairies</font>

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.warn('DelftStack')
warnings.warn('Do not show this message')

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go

import kagglehub

from scipy.stats import gaussian_kde

import sys
import os

# <font color='#3585CD'>Fonctions qui seront utilisées dans ce notebook</font>

In [ ]:
 # variable de visualisation : 
# si visualisation == plotly alors les graphiques seront intéractifs
# si visualisation == seaborn alors les graphiques seront statiques
visualisation = 'seaborn'

def display_missing_values(df):
  """
  Affiche un DataFrame contenant les valeurs manquantes, leur pourcentage / nombre et leur type, pour chaque colonne du DataFrame.

  :param df: DataFrame contenant la variable à analyser.
  """
  missing_values = df.isnull().sum()
  missing_ratio = (missing_values / len(df)) * 100

  missing_values_df = pd.DataFrame({
      'Colonne': missing_values.index,
      'Valeurs manquantes (%)': missing_ratio.values,
      'Nombre de valeurs manquantes': missing_values.values,
      'Type': df.dtypes.values
  })

  missing_values_df = missing_values_df[missing_values_df['Nombre de valeurs manquantes'] > 0]
  missing_values_df = missing_values_df.sort_values(by='Valeurs manquantes (%)', ascending=False).reset_index(drop=True)

  return missing_values_df

def calculer_correlation(df, col1, col2):
  """
  Calcule et affiche la corrélation entre deux colonnes d'un DataFrame.

  Paramètres :
  df :DataFrame contenant les données.
  col1 : nom de la première colonne.
  col2 : nom de la deuxième colonne.

  Retourne la valeur de la corrélation et une interprétation de son intensité.
  """
  correlation = df[col1].corr(df[col2])
  print(f"Corrélation entre {col1} et {col2} : {correlation:.2f}")

  if abs(correlation) > 0.8:
      interpretation = "Très forte corrélation"
  elif abs(correlation) > 0.6:
      interpretation = "Forte corrélation"
  elif abs(correlation) > 0.4:
      interpretation = "Corrélation modérée"
  elif abs(correlation) > 0.15:
      interpretation = "Corrélation faible"
  else:
      interpretation = "Pas de corrélation linéaire significative"

  print(interpretation + ".\n")

  return correlation, interpretation

def analyse_columns(df):

    """
    Analyse les colonnes d'un DataFrame : nom, type, nombre de valeurs uniques et exemples de valeurs.

    :param df: DataFrame Pandas
    
    Retourne un DataFrame avec l'analyse des colonnes
    """
    analysis = []

    for col in df.columns:
        col_name = col
        col_type = df[col].dtype
        unique_count = df[col].nunique()  # Nombre de valeurs uniques
        unique_values = df[col].dropna().unique()[:5].tolist()  # Exemples (max 5)
        unique_values = " | ".join(map(str, unique_values))  # Séparateur : " | "

        analysis.append({
            'Nom de la colonne': col_name,
            'Type de la colonne': col_type,
            'Nombre de valeurs uniques': unique_count,
            'Exemples de valeurs': unique_values
        })

    return pd.DataFrame(analysis)

def analyser_variable_categorielle(df, variable, top_n=100, display_array=True, viz=visualisation):
    """
    Analyse une variable catégorielle en affichant un DataFrame des 'top_n' catégories les plus fréquentes,
    ainsi qu'un graphique Plotly ou Seaborn.

    :param df: DataFrame contenant la variable à analyser.
    :param variable: Nom de la variable catégorielle.
    :param top_n: Nombre de catégories à afficher (par défaut 100).
    :param display_array: Booléen pour afficher le tableau résumé.
    :param viz: 'plotly' ou 'seaborn'.
    """
    if display_array:
        top_cat = f"(Top {top_n} catégories)" if top_n != 100 else ""
        print(f"\nAnalyse de la variable : {variable} {top_cat}")

    # Calcul des fréquences et pourcentages
    category_counts = df[variable].value_counts().head(top_n)
    category_percent = df[variable].value_counts(normalize=True).head(top_n) * 100

    # Création du DataFrame de synthèse
    df_summary = pd.DataFrame({
        "Libellé": category_counts.index,
        "Total": category_counts.values,
        "Pourcentage": category_percent.values
    })

    # Affichage du tableau
    if display_array:
        display(df_summary)

    # Affichage du graphique
    if viz == 'plotly':
        nb = top_n if top_n != 100 else ""
        fig = px.bar(df_summary,
                     x="Libellé",
                     y="Total",
                     text="Total",
                     title=f'Distribution des {nb} premières catégories de {variable}',
                     labels={"Libellé": variable, "Total": "Nombre d\'occurrences"},
                     template="plotly_white")
        fig.update_traces(textposition='outside')
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

    elif viz == 'seaborn':
        plt.figure(figsize=(10, 6))
        sns.barplot(data=df_summary, x="Libellé", y="Total", palette="viridis")
        plt.title(f'Distribution des {top_n} premières catégories de {variable}')
        plt.xlabel(variable)
        plt.ylabel("Nombre d'occurrences")
        plt.xticks(rotation=45, ha='right')
        # for index, value in enumerate(df_summary["Total"]):
        #     plt.text(index, value, str(value), ha='center', va='bottom')
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

def analyser_variables_numeriques(df, variables, bins=30, viz=visualisation):
    """
    Analyse les variables numériques en affichant un histogramme + KDE (distribution) et un boxplot interactifs.

    :param df: DataFrame contenant les données.
    :param variables: Liste des variables numériques à analyser.
    :param bins: Nombre de bins pour l'histogramme (par défaut 30).
    :param viz: 'plotly' ou 'seaborn'.
    """
    for var in variables:
        data = df[var].dropna()

        if viz == "plotly":
            hist = go.Histogram(
                x=data,
                nbinsx=bins,
                marker=dict(color='skyblue', line=dict(color='black', width=1)),
                opacity=0.6,
                name="Histogramme"
            )

            kde = gaussian_kde(data)
            x_vals = np.linspace(data.min(), data.max(), 500)
            kde_vals = kde(x_vals)

            kde_curve = go.Scatter(
                x=x_vals,
                y=kde_vals * len(data) * (data.max() - data.min()) / bins,
                mode='lines',
                line=dict(color='blue', width=2),
                name="Densité (KDE)"
            )

            fig = go.Figure(data=[hist, kde_curve])
            fig.update_layout(
                title=f'Distribution de {var} (Histogramme + KDE)',
                xaxis_title="Valeur",
                yaxis_title="Fréquence",
                template="plotly_white",
                barmode='overlay'
            )
            fig.show()

        elif viz == "seaborn":
            plt.figure(figsize=(12,6))
            sns.histplot(data, bins=bins, kde=True, color='skyblue', edgecolor='black')
            plt.title(f'Distribution de {var} (Histogramme + KDE)')
            plt.xlabel("Valeur")
            plt.ylabel("Fréquence")
            plt.tight_layout()
            plt.show()

        else:
            raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")


def analyser_hist_co2_par_variable(df, variable, top_n=50, order='desc', co2="CO2", viz=visualisation):
    """
    Génère un histogramme avec une ligne de moyenne CO2.

    Paramètres :
    :param df : DataFrame contenant les données
    :param variable : Nom de la colonne à analyser
    :param top_n: Nombre de catégories à afficher (par défaut 50).
    :param order: Ordonnancement des catégories (par défaut 'desc').
    :param co2: Nom de la colonne des émissions de CO2 (par défaut "CO2")
    :param viz: 'plotly' ou 'seaborn'.
    """

    ascending = True if order == 'asc' else False

    df_co2 = df.groupby(variable)[co2].mean().sort_values(ascending=ascending).reset_index()
    df_co2 = df_co2.head(top_n)

    df_co2['CO2_txt'] = df_co2[co2].apply(lambda x: f"{x:.2f}")
    moyenne_co2 = df[co2].mean()

    if viz == 'plotly':
        fig = px.bar(
            df_co2,
            x=variable,
            y=co2,
            title=f"Distribution des émissions de CO2 par {variable} (Top {top_n})",
            labels={variable: variable.capitalize(), co2: "Émissions de CO2 (g/km)"},
            color=co2,
            color_continuous_scale="RdYlGn_r",
            text='CO2_txt'
        )

        fig.add_hline(
            y=moyenne_co2,
            line_dash="dot",
            line_color="red",
            annotation_text=f"Moyenne CO2: {moyenne_co2:.2f} g/km",
            annotation_position="top right",
            annotation_font_color="red",
            annotation_font_size=12,
            annotation_bgcolor="rgba(255,255,255,0.7)"
        )

        fig.update_layout(
            xaxis_tickangle=-75,
            coloraxis_colorbar=dict(title="CO2 (g/km)"),
            uniformtext_minsize=8,
            uniformtext_mode='hide'
        )

        fig.show()

    elif viz == 'seaborn':
        plt.figure(figsize=(12,6))
        sns.barplot(data=df_co2, x=variable, y=co2, palette='coolwarm')
        plt.axhline(moyenne_co2, color='red', linestyle='--', label=f"Moyenne CO2: {moyenne_co2:.2f}")
        plt.title(f"Distribution des émissions de CO2 par {variable} (Top {top_n})")
        plt.xticks(rotation=75, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

def plot_scatter_co2(df, x, y="CO2", color="Carburant", size="CO2", viz=visualisation):
    """
    Génère un scatter plot avec une ligne de moyenne CO2.

    Paramètres :
    - df : DataFrame contenant les données
    - x : Nom de la colonne pour l'axe X
    - y : Nom de la colonne pour l'axe Y (par défaut "CO2")
    - color : Nom de la colonne pour la couleur des points (par défaut "Carburant")
    - size : Nom de la colonne pour la taille des points (par défaut "CO2")
    - param viz: 'plotly' ou 'seaborn'.
    """

    moyenne_co2 = df[y].mean()

    if viz == 'plotly':
        fig = px.scatter(
            df,
            x=x,
            y=y,
            color=color,
            size=size,
            title=f"Relation entre {x} et {y}",
            labels={x: x.capitalize(), y: y.capitalize(), color: color.capitalize()},
            hover_data=df.columns,
            size_max=20
        )

        fig.add_hline(
            y=moyenne_co2,
            line_dash="dot",
            line_color="red",
            annotation_text=f"Moyenne CO2: {moyenne_co2:.2f} g/km",
            annotation_position="top right",
            annotation_font_color="red",
            annotation_font_size=12,
            annotation_bgcolor="rgba(255,255,255,0.7)"
        )

        fig.show()

    elif viz == 'seaborn':
        plt.figure(figsize=(10,6))
        sns.scatterplot(data=df, x=x, y=y, hue=color, alpha=0.6)
        plt.axhline(moyenne_co2, color='red', linestyle='--', label=f"Moyenne CO2: {moyenne_co2:.2f}")
        plt.title(f"Relation entre {x} et {y}")
        plt.xlabel(x)
        plt.ylabel(y)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

def plot_correlation_matrix(df, viz=visualisation):
    """
    Affiche la matrice de corrélation des variables numériques sous forme de heatmap.

    :param df: DataFrame Pandas contenant les données
    :param viz: 'plotly' ou 'seaborn'.
    """

    num_numeric_cols = df.select_dtypes(include=['number']).columns
    corr_matrix = df[num_numeric_cols].corr()

    if viz == 'plotly':
        fig = ff.create_annotated_heatmap(
            z=corr_matrix.values,
            x=list(corr_matrix.columns),
            y=list(corr_matrix.index),
            colorscale="RdBu_r",
            annotation_text=corr_matrix.round(2).values,
            showscale=True
        )

        fig.update_layout(
            title="Matrice de corrélation des variables numériques",
            height=600, width=800,
            xaxis=dict(side="bottom", tickangle=-45),
            yaxis=dict(side="left")
        )

        fig.show()

    elif viz == 'seaborn':
        plt.figure(figsize=(12,10))
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", linewidths=0.5)
        plt.title("Matrice de corrélation des variables numériques")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

def detecter_outliers(serie, seuil=1.5, viz=visualisation):
    """
    Détecte les outliers pour une seule variable numérique en utilisant la méthode IQR.
    Affiche également un Boxplot.

    :param serie: Série Pandas contenant les valeurs de la variable.
    :param seuil: Seuil du coefficient IQR (par défaut 1.5).
    :return: DataFrame contenant le nombre d'outliers et le pourcentage.
    :param viz: 'plotly' ou 'seaborn'.
    """

    if not isinstance(serie, pd.Series):
        raise ValueError("Veuillez fournir une variable sous forme de pd.Series")

    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - seuil * IQR
    upper_bound = Q3 + seuil * IQR
    outliers = serie[(serie < lower_bound) | (serie > upper_bound)]

    nb_outliers = outliers.shape[0]
    perc_outliers = (nb_outliers / serie.shape[0]) * 100

    if viz == 'plotly':
        fig = go.Figure()
        fig.add_trace(go.Box(
            x=serie,
            name=serie.name if serie.name else "Variable",
            marker_color='blue',
            boxpoints='outliers'
        ))
        fig.update_layout(
            title=f"Box Plot de {serie.name if serie.name else 'Variable'}",
            xaxis_title="Valeur",
            yaxis_title="Variable",
            template="plotly_white",
            showlegend=False
        )
        fig.show()

    elif viz == 'seaborn':
        plt.figure(figsize=(8,4))
        sns.boxplot(x=serie, color='skyblue')
        plt.title(f"Box Plot de {serie.name if serie.name else 'Variable'}")
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

    df_outliers = pd.DataFrame({
        "Nb_Outliers": [nb_outliers],
        "Pourcentage": [round(perc_outliers, 2)]
    }, index=[serie.name if serie.name else "Variable"])
    
    return df_outliers

def analyser_hist_co2_par_variable(df, variable, top_n=50, order='desc', co2="CO2", viz=visualisation):
    """
    Génère un histogramme des émissions de CO2 moyen par modalité d'une variable.

    :param df: DataFrame contenant les données
    :param variable: Nom de la colonne à analyser
    :param top_n: Nombre de catégories à afficher (par défaut 50)
    :param order: Ordre de tri ('asc' ou 'desc')
    :param co2: Nom de la colonne contenant les émissions de CO2
    :param viz: Bibliothèque de visualisation ('plotly' ou 'seaborn')
    """
    ascending = True if order == 'asc' else False

    df_co2 = df.groupby(variable)[co2].mean().sort_values(ascending=ascending).reset_index()
    df_co2 = df_co2.head(top_n)
    df_co2['CO2_txt'] = df_co2[co2].apply(lambda x: f"{x:.2f}")
    moyenne_co2 = df[co2].mean()

    if viz == 'plotly':
        fig = px.bar(
            df_co2,
            x=variable,
            y=co2,
            title=f"Distribution des émissions de CO2 par {variable} (Top {top_n})",
            labels={variable: variable.capitalize(), co2: "Émissions de CO2 (g/km)"},
            color=co2,
            color_continuous_scale="RdYlGn_r",
            text='CO2_txt'
        )

        # Ligne de moyenne du CO2
        fig.add_hline(
            y=moyenne_co2,
            line_dash="dot",
            line_color="red",
            annotation_text=f"Moyenne CO2: {moyenne_co2:.2f} g/km",
            annotation_position="top right",
            annotation_font_color="red",
            annotation_font_size=12,
            annotation_bgcolor="rgba(255,255,255,0.7)"
        )

        fig.update_layout(
            xaxis_tickangle=-75,
            coloraxis_colorbar=dict(title="CO2 (g/km)"),
            uniformtext_minsize=8,
            uniformtext_mode='hide'
        )
        fig.show()

    elif viz == 'seaborn':
        ordered_df = df.groupby(variable)[co2].mean().sort_values(ascending=ascending).reset_index()
        ordered_df = ordered_df.head(top_n)

        # Palette toujours dans le même sens (rouge = CO2 élevé)
        norm = plt.Normalize(ordered_df[co2].min(), ordered_df[co2].max())
        cmap = cm.get_cmap('RdYlGn_r')  # toujours inversée pour que le rouge = haut CO2

        # Couleurs par valeur
        colors = [cmap(norm(value)) for value in ordered_df[co2]]

        plt.figure(figsize=(12, 6))
        ax = sns.barplot(data=df_co2, x=variable, y=co2, palette=colors)
        plt.title(f"Distribution des émissions de CO2 par {variable} (Top {top_n})")
        plt.xlabel(variable)
        plt.ylabel("Émissions de CO2 (g/km)")
        plt.xticks(rotation=75, ha='right')

        # for index, value in enumerate(df_co2[co2]):
        #     ax.text(index, value, f"{value:.2f}", ha='center', va='bottom')

        # Ligne de moyenne
        plt.axhline(moyenne_co2, linestyle='--', color='red', label=f"Moyenne CO2: {moyenne_co2:.2f} g/km")
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("Le paramètre 'viz' doit être 'plotly' ou 'seaborn'.")

# <font color='#3585CD'>Chargement des données</font>

## Chargement du dataset principal

Le dataset étant **volumineux** pour être stocké dans le repository Github, nous le téléchargeons via **Kaggle**

In [ ]:
path = kagglehub.dataset_download("dimitrileloup/vehicules-fr-2022-2023")
print("Chemin vers le fichier : ", path)

In [ ]:
dataset_path = f"{path}/datas_FR_2022_2023.csv"
df = pd.read_csv(dataset_path)
df.head(10)

In [ ]:
# Suppression des espaces "accidentels" dans les noms des colonnes comme "Fuel consumption "
df.columns = df.columns.str.strip()

## Chargement du dataset du dictionnaire des variables

Même si le fichier du dictionnaire n'est pas autant volumineux que le dataset, nous décidons aussi de le télécharger via Kaggle.

In [ ]:
path_vars = kagglehub.dataset_download("dimitrileloup/dfinition-des-colonnes")
print("Path to dataset files:", path_vars)

In [ ]:
pd.set_option('display.max_colwidth', None) # pour pouvoir afficher tout le descriptif
dataset_variables = f"{path_vars}/Table-definition.xlsx"
var = pd.read_excel(dataset_variables)
var.head(40)

# <font color='#3585CD'>Premières analyses</font>

## Informations sur le dataset

In [ ]:
print("\nAperçu du dataset :")
print(df.info())

Le dataset est composé de **3 528 480** lignes et **40** colonnes.

## Statistiques descriptives

### Variables numériques

In [ ]:
print("\nStatistiques descriptives :")
display(df.describe(include='number').T)

### Variables catégorielles

In [ ]:
df.describe(include="object").T

## Analyse rapide des colonnes
La fonction `analyse_columns` permet de visualiser les différentes colonnes avec leur type, leur nombre de valeurs uniques et un échantillon de valeurs.

In [ ]:
analysis_df = analyse_columns(df)
analysis_df

# <font color='#3585CD'>Analyse des valeurs manquantes</font>

La fonction `display_missing_values` permet de visualiser les valeurs manquantes du dataset.

In [ ]:
data_na = display_missing_values(df)
data_na

**8 colonnes** ont des valeurs manquantes avec un taux **supérieur à 70 %**.

**4 colonnes** ont des valeurs manquantes avec un taux **supérieur à 49 %**.


# <font color='#3585CD'>Suppression de colonnes</font>

Nous pouvons dès à présent **supprimer** les colonnes qui ont un taux de valeurs manquantes **supérieur à 70%** :

*   MMS
*   Vf
*   De
*   Ernedc (g/km)
*   Enedc (g/km)
*   RLFI
*   Electric range (km)
*   z (Wh/km)

In [ ]:
df = df.drop(columns=['MMS', 'Vf', 'De', 'Ernedc (g/km)', 'Enedc (g/km)', 'RLFI', 'z (Wh/km)', 'Electric range (km)'], axis=1)

In [ ]:
data_na = display_missing_values(df)
data_na

# <font color='#3585CD'>Suppression des colonnes non pertinentes</font>

Nous décidons de supprimer les colonnes suivantes, ne les jugeant pas pertinentes pour notre projet :

*   At2 (mm)
*   W (mm)
*   At1 (mm)
*   IT
*   Erwltp (g/km) (nous souhaitons prédire Ewltp (g/km))
*   ID : identifiant du véhicule
*   Country : notre dataset est une extraction des véhicules de France
*   VFN : n'a pas de norme universelle et comporte trop de valeurs
*   Tan : trop de valeurs et sans intérêt pour notre projet
*   T : trop de valeurs et sans intérêt pour notre projet
*   Va : trop de valeurs et sans intérêt pour notre projet
*   Ve : trop de valeurs et sans intérêt pour notre projet
*   Status : n'a qu'une seule valeur et ne varie pas
*   Year : 1 seule valeur
*   Date of registration : sans intérêt pour notre projet
*   Fm : redondant avec Ft
*   Cr : nous avons 2 catégories (M1, M1G). M1G est une sous-catégorie de M1 réservée aux véhicules tout-terrain. Nous pouvons conclure que tous les véhicules sont de catégorie M1
*   Ct : idem que Cr
*   ech : sans intérêt pour notre projet
*   Mp : redondant, se retrouve dans une autre colonne
*   Man : redondant avec Mk
*   r : n'a qu'une seule valeur
*   Mh : redondant avec Mk


In [ ]:
df = df.drop(columns=['At2 (mm)', 'W (mm)', 'At1 (mm)', 'IT', 'Erwltp (g/km)', 'ID', 'Country', 'VFN', 'Tan', 'T', 'Va', 'Ve', 'Status', 'year', 'Date of registration', 'Fm', 'Cr', 'Ct', 'ech', 'Mp', 'Man', 'r', 'Mh'], axis=1)

## Vérification de la corrélation entre Masse à vide et Masse totale</font>

In [ ]:
df_masse = df[['Mt', 'm (kg)']]
plot_correlation_matrix(df_masse)

On se rend compte qu'il y a une **forte corrélataion** entre ces 2 variables, qui pourrait entrainer une **colinéarité**. Nous prenons la décision de ne garder que la masse à vide (m (kg)).

In [ ]:
df = df.drop('Mt', axis=1)

# <font color='#3585CD'>Gestion des doublons</font>

## Nombre de doublons

In [ ]:
print(f"Nombre de doublons : {df.duplicated().sum()}")

## Suppression des doublons

In [ ]:
# Suppression des doublons
df.drop_duplicates(inplace =True)

In [ ]:
# Vérification qu'il n'y ait plus de doublons
print(f"Nombre de doublons : {df.duplicated().sum()}")

In [ ]:
# Nombres de lignes restantes
print(f"Il reste {df.shape[0]} lignes dans notre dataset.")

In [ ]:
# Ré initialisation des index
df = df.reset_index(drop=True)

# <font color='#3585CD'>Renommage des colonnes</font>

Pour plus de compréhension, nous allons renommer les colonnes suivantes :



*   Mk : Marque
*   Cn : Modele
*   Ewltp (g/km) : CO2
*   Ft : Carburant
*   ec (cm3) : Cylindrée moteur
*   ep (KW) : Puissance moteur
*   Fuel consumption : Consommation carburant


In [ ]:
renommage = {
    'Mk': 'Marque',
    'Cn': 'Modèle',
    'm (kg)' : 'Masse à vide',
    'Ewltp (g/km)': 'CO2',
    'Ft': 'Carburant',
    'ec (cm3)': 'Cylindrée moteur',
    'ep (KW)': 'Puissance moteur',
    'Fuel consumption': 'Consommation carburant'
}

# Application du renommage
df.rename(columns=renommage, inplace=True)

In [ ]:
df.head(10)

# <font color='#3585CD'>Faut-il garder les véhicules électriques et hydrogènes ?</font>
Notre objectif est de **prédire les émissions directes de CO2**. Garder les véhicules électriques et hydrogènes risque de biaiser notre modèle. Nous allons donc **exclure les véhicules électriques et hydrogènes** de notre dataset.

In [ ]:
# Vérification des émissions de CO2 des véhicules hydrogènes avant exclusion
df_hydrogen = df[df["Carburant"] == "hydrogen"]
df_hydrogen['CO2'].value_counts()

In [ ]:
# Nous excluons les véhicules électriques et hydrogènes
df = df[(df["Carburant"] != "electric") & (df["Carburant"] != "hydrogen")]

In [ ]:
# Nombres de lignes restantes
print(f"Il reste {df.shape[0]} lignes dans notre dataset.")

# <font color='#3585CD'>Traitement des valeurs manquantes</font>

In [ ]:
data_na = display_missing_values(df)
data_na

Il nous reste 2 variables à traiter.

## Gestion des valeurs manquantes de la colonne Consommation carburant

Regardons la ligne dont la variable Consommation carburant est nulle :

In [ ]:
df_fc_na = df[df['Consommation carburant'].isna()]
df_fc_na

Nous allons gérer cette ligne en recherchant des modèles équivalents dans notre dataset :

In [ ]:
# regardons si d'autres modeles RANGE ROVER EVOQUE sont renseignés
df_jag_evoque = df[(df['Modèle'] == 'RANGE ROVER EVOQUE') & (df['Carburant'] == 'diesel') & (~df['Consommation carburant'].isna())]
df_jag_evoque.head()

Plusieurs modèles ayant les mêmes caractéristiques sont présents dans notre dataset. Nous pouvons remplacer la valeur manquante par la **moyenne** de cette même variable des modèles similaires : 

In [ ]:
# Nous récupérons la consommation de carburant moyenne pour le même modèle
fc_mean_jag_evoque = df_jag_evoque['Consommation carburant'].mean().round(1)
print(f"La moyenne de la consommation de carburant est de {fc_mean_jag_evoque}")

In [ ]:
# nous attribuons la valeur fc_mean_jag_evoque
df.loc[(df["Modèle"] == "RANGE ROVER EVOQUE") & (df['Consommation carburant'].isna()), "Consommation carburant"] = fc_mean_jag_evoque

In [ ]:
# vérification des valeurs manquantes
data_na = display_missing_values(df)
data_na

## Gestion des valeurs manquantes de la colonne Masse à vide

In [ ]:
data_na = display_missing_values(df)
data_na

Regardons la ligne dont la variable Masse à vide est nulle :

In [ ]:
# récupération de la ligne dont la masse est égale à NaN
df_mkg_na = df[df['Masse à vide'].isna()]
df_mkg_na

Recherchons dans le dataset si nous avons des modèles équivalents dont la variable Masse à vide n'est **pas nulle** : 

In [ ]:
df_jag_lr = df[(df['Modèle'] == 'RANGE ROVER EVOQUE') & (df['Carburant'] == 'diesel') & (df['Puissance moteur'] == 120) & (~df['Masse à vide'].isna())]
df_jag_lr

Plusieurs modèles ayant les mêmes caractéristiques sont présents dans notre dataset. Nous pouvons remplacer la valeur manquante par la **moyenne** de cette même variable des modèles similaires : 

In [ ]:
# Nous récupérons la masse moyenne pour le même modèle
fc_mean_masse_jag_evoque = df_jag_evoque['Masse à vide'].mean().round(1)
print(f"La moyenne de la masse est de {fc_mean_masse_jag_evoque}")

In [ ]:
# nous attribuons la valeur fc_mean_masse_jag_evoque
df['Masse à vide'] = df['Masse à vide'].fillna(fc_mean_masse_jag_evoque)

In [ ]:
# Vérification de l'assignation de la valeur
df.loc[23942].to_frame().T

In [ ]:
data_na = display_missing_values(df)
data_na

Nous n'avons **plus de valeurs manquantes** dans notre dataset.

In [ ]:
# ré inisialisation des indexes
df = df.reset_index(drop=True)
df.head()

# <font color='#3585CD'>Distribution des variables catégorielles</font>

## Analyse par marque

### Valeurs uniques

In [ ]:
sorted(df['Marque'].unique())

### Remplacement

Certaines valeurs peuvent être regroupées, comme par exemple :

*   'MC LAREN', 'MCLAREN'
*   'MERCEDES AMG', 'MERCEDES BENZ', 'MERCEDES-BENZ'
*   'MITSUBISHI', 'MITSUBISHI MOTORS CORPORATION', 'MITSUBISHI MOTORS THAILAND'


In [ ]:
# valeurs à remplacer
replace_mk = {'MC LAREN' : 'MCLAREN',
              'MERCEDES AMG' : 'MERCEDES BENZ',
              'MERCEDES-BENZ' : 'MERCEDES BENZ',
              'MITSUBISHI MOTORS CORPORATION' : 'MITSUBISHI',
              'MITSUBISHI MOTORS THAILAND' : 'MITSUBISHI',
              'MITSUBISHI MOTORS (THAILAND)' : 'MITSUBISHI',
              'FORD-CNG-TECHNIK' : 'FORD',
              'ROLLS ROYCE' : 'ROLLS-ROYCE'}
df['Marque'] = df['Marque'].replace(replace_mk)

In [ ]:
print(f"Il y a {df['Marque'].nunique()} marques dans le dataset.")

### Analyse

In [ ]:
analyser_variable_categorielle(df, 'Marque', 30, False)

Sans surpise, nous nous apercevons que les marques les **plus représentées** sont les **plus connues** : BMW, Mercedes, Skoda, Audi, Volswagen, Ford, Peugeot ...

## Analyse par carburant

### Valeurs uniques

In [ ]:
sorted(df['Carburant'].unique())

### Remplacement

Certaines valeurs peuvent être regroupées :

*   'diesel/electric' & 'petrol/electric' sont des véhicules hybrides. Nous décidons de créer une catégorie 'hybride' afin de les regrouper.
*   'lpg' & 'ng' sont des énergies au gaz. Nous décidons de créer une catégorie 'gaz' afin de les regrouper.
*   'petrol' sera renommé en 'essence' pour plus de compréhension.
*   'e85' est un carburant de type 'essence'. Nous décidons de "l'absorber" dans la catégorie 'essence'.


In [ ]:
replace_ft = {
              'petrol' : 'essence',
              'diesel/electric' : 'hybride',
              'petrol/electric' : 'hybride',
              'lpg' : 'gaz',
              'ng' : 'gaz',
              'e85' : 'essence'}
df['Carburant'] = df['Carburant'].replace(replace_ft)

### Analyse

In [ ]:
analyser_variable_categorielle(df=df, variable='Carburant', display_array=False)

Les carburants majoritaires sont : 
*   Essence
*   Diesel
*   Hybride

## Analyse par modèle de voiture

In [ ]:
analyser_variable_categorielle(df, 'Modèle', 30, display_array=False)

Sans surprise, les modèles les plus nombreux sont des **"classiques"** du monde automobile : Octovia et Kodiaq de Skoda, Focus de Ford, Golf de Volkswagen ...

# <font color='#3585CD'>Distribution des variables numériques</font>

## Sélection des colonnes numériques

In [ ]:
# Sélection des colonnes numériques
num_vars = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_vars

## Analyse de la masse du véhicule Masse à vide

In [ ]:
analyser_variables_numeriques(df, ['Masse à vide'])

La classe est entre **1400 et 1700 kg** contient la plus grande concentration de véhicules.

## Analyse des émissions spécifiques de CO2

In [ ]:
analyser_variables_numeriques(df, ['CO2'], 30)

Nous observons un **pic massif** entre **100 et 150 g/km**, signifiant que la majorité des véhicules se trouvent dans cette fourchette.

Nous observons également un **pic secondaire** plus petit entre **20 et 50 g/km**, probablement des véhicules hybrides.

Enfin , une **longue traîne à droite** est visible, signifiant que quelques véhicules **très polluants** sont présents dans notre dataset (SUV, utilitaires, sportives).

## Analyse de la cylindrée moteur

In [ ]:
analyser_variables_numeriques(df, ['Cylindrée moteur'], 50)

Il y a plusieurs **pics nets** : 

* Autour de 1200 cm³
* Puis 1600 cm³,
* Un gros pic vers 1900 cm³
* Un 4e petit pic vers 3000 cm³, 
* Et quelques cas rares au-delà de 4000 cm³.

Peut-être segmenter cette colonne ? 
* < 1300 : Petite cylindrée	
* 1300 – 1599 : Cylindrée Moyenne
* 1600 – 1999: Cylindrée standard	
* 2000 – 2999 : Grosse cylindrée
* 3000 et plus : Très grosse cylindrée

In [ ]:
bins = [0, 1299, 1599, 1999, 2999, float('inf')]
labels = [
    'Petite (≤1.3L)',
    'Moyenne (1.3–1.6L)',
    'Standard (1.6–2.0L)',
    'Grosse (2.0–3.0L)',
    'Très grosse (>3.0L)'
]

df['Classe_Cylindree'] = pd.cut(df['Cylindrée moteur'], bins=bins, labels=labels, right=True)


In [ ]:
df.head()

In [ ]:
analyser_variable_categorielle(df, 'Classe_Cylindree')

## Analyse de la puissance du moteur

In [ ]:
analyser_variables_numeriques(df, ['Puissance moteur'])

La majorité des véhicules ont une puissance entre **50 et 150 ch**.

Le pic majeur est clairement autour de **100 ch**.

Ensuite, la fréquence chute fortement à partir de **150–160 ch**, mais il reste une queue étalée jusqu’à plus de 400 ch.

Peut-être segmenter cette colonne ? 
* < 110 ch : faible
* 110 – 149 ch : moyenne
* 150 – 199 ch : intermédiaire
* 200 - 299 ch: élevée
* sup. à 300 ch : très élevée

In [ ]:
bins = [0, 109, 149, 199, 299, float('inf')]
labels = [
    'Faible (<110 ch)',
    'Moyenne (110–149 ch)',
    'Performante (150–199 ch)',
    'Élevée (200–299 ch)',
    'Très élevée (≥300 ch)'
]

df['Classe_Puissance'] = pd.cut(df['Puissance moteur'], bins=bins, labels=labels, right=True)

In [ ]:
df.head()

In [ ]:
analyser_variable_categorielle(df, 'Classe_Puissance')

## Analyse de la Consommation carburant

In [ ]:
analyser_variables_numeriques(df, ['Consommation carburant'])

La consommation la plus fréquente se situe entre **5 et 6 L/100 km**, avec un pic net vers **5.5 L**.

Il y a quelques véhicules **extrêmement économes** (<3 L), probablement des véhicules hybrides et une **longue traîne à droite** jusqu’à 25 L/100 km, probablement des véhicules sportifs.

# <font color='#3585CD'>Création d'indicateurs</font>

Nous souhaitons créer quelques indicateurs afin de faciliter l'analyse et l'interprétation. Nous déciderons lors de la modélisation si nous les gardons ou pas.

## Indicateur de Charge Spécifique du Moteur (ICSM)

`ICSM = Puissance (kW) / Masse du véhicule (kg)`

**Interprétation** :

*   Faible ICSM → Voiture puissante et légère (moins d’effort, moins de CO₂).
*   Élevé ICSM → Voiture sous-motorisée (forte sollicitation, plus de CO₂).

In [ ]:
df['ICSM'] = df['Puissance moteur'] / df['Masse à vide']

## Indicateur de Consommation Énergétique (ICE)

`ICE = Puissance (kW) / Cylindrée (cm³)`

**Interprétation** :

*   Faible ICE → Moteur optimisé (ex. turbo downsizing).
*   Élevé ICE → Moteur gourmand et peu efficient.

In [ ]:
df['ICE'] = df['Puissance moteur'] / df['Cylindrée moteur']

## Indicateur de Densité Energétique du Carburant (IDEC)

`IDEC = Cylindrée (cm³) / Masse du véhicule (kg)`

**Interprétation** :

*   Faible IDEC → Moteur bien dimensionné (moins d’effort, moins de CO₂).
*   Élevé IDEC → Moteur sous-dimensionné (forte sollicitation, plus de CO₂).

In [ ]:
df['IDEC'] = df['Cylindrée moteur'] / df['Masse à vide']

## Indicateur de Consommation Spécifique (ICS)

`ICS = Puissance produite (kWh) / Consommation de carburant (g)`

**Interprétation** :

*   Faible ICS : Moteur efficace, consomme moins de carburant et génère moins de CO₂ pour produire la même puissance.
*   Élevé ICS : Moteur inefficace, consomme plus de carburant et génère plus de CO₂ pour produire la même puissance.

In [ ]:
df['ICS'] = df['Puissance moteur'] / df['Consommation carburant']

In [ ]:
df.head()

# <font color='#3585CD'>Analyse du CO2 en fonction de certaines variables</font>

### Analyse du CO2 en fonction de la masse du véhicule

In [ ]:
plot_scatter_co2(df, "Masse à vide")

Nous observons une **corrélation positive** entre la masse et le CO2, ce qui peut être logique : plus un véhicule est lourd, plus il consomme de carburant et donc plus il émet de de CO2.

Cependant, les véhicules Hybrides **"cassent"** cette logique : malgré une masse élevée les émissions de CO2 restent faibles.

Les véhicules Essence tendent à être **plus émissif** que les véhicules Diesel à masse équivalente.

### Analyse du CO2 en fonction de la puissance du moteur.

In [ ]:
plot_scatter_co2(df, "Puissance moteur")

Nous observons une **corrélation positive** entre la puissance et le CO2 : plus un véhicule est puissant, plus il émet de CO2.

Les véhicules Hybrides sont clairement en-dessous de la ligne moyenne des émissions de CO2.

De très rares valeurs extrêmes sont présentes.

### Analyse du CO2 en fonction du carburant



In [ ]:
analyser_hist_co2_par_variable(df, 'Carburant')

Les véhicules Hybrides sont **nettement en dessous** que tous les autres carburants en matière d'émissions de CO2.

Les véhicules Diesel et Essence sont très **proches** (environ 150 g/km), et tous **au-dessus** de la moyenne.

Le Gaz naturel est une **alternative intéressante** qui est sous la moyenne.

### Analyse du CO2 en fonction de la marque

**Top des marques les plus polluantes**

In [ ]:
analyser_hist_co2_par_variable(df, 'Marque', 25)

La marque **Bugatti** est largement au dessus des autres marques avec plus de 500 g/km (sans surprise vu le type de véhicule).

Les marques **sportives/premium** (Lamborghini, Ferrari, McLaren...) dominent le haut du classement.

**Top des marques les moins polluantes**

In [ ]:
analyser_hist_co2_par_variable(df, 'Marque', 25, order='asc')

La marque **LYNK&CO** est largement en tête, probablement car sa flotte est quasi uniquement hybride.

La marque **MG** est aussi faible émettrice de CO2.

Les marques **européennes classiques** (Peugeot, Renault, Volkswagen) se situent autour de 125–140 g/km.

### Analyse du CO2 en fonction du modèle de voiture

**Top des modèles les plus polluants**

In [ ]:
analyser_hist_co2_par_variable(df, 'Modèle', 20)

Ce Top 20 ne contient que des véhicules **"ultra-premium"** : Bugatti, Ferrari, Lamborghini, AMG, Rolls-Royce…

Ce sont tous des modèles très puissants, lourds, donc naturellement **très émissifs**.

La différence énorme avec la moyenne (140.72) montre à quel point ces modèles sont hors norme d'un point de vue environnementale.

**Top des modèles les moins polluants**

In [ ]:
analyser_hist_co2_par_variable(df, 'Modèle', 20, order='asc')

Ces valeurs extrêmement faibles (entre 12 et 20 g/km) sont caractéristiques des **véhicules hybrides**, qui combinent moteur électrique/thermique.

Les marques Mercedes, BMW, Toyota dominent ce classement.

### Analyse du CO2 en fonction de la consommation

In [ ]:
plot_scatter_co2(df, "Consommation carburant")

Nous observons une **relation quasi-linéaire** entre la consommation de carburant et le CO2 : plus un véhicule consomme de carburant, plus il émet de CO2.

Les véhicules Diesel ont tendance à **emettre moins de CO2** par rapport aux véhicules Essance.

Les véhicules Hybrides sont **concentrés en bas à gauche** : consommation de carburant faible et émissions de CO2 faibles.

### Analyse du CO2 en fonction de la classe de Puissance


In [ ]:
analyser_hist_co2_par_variable(df, 'Classe_Puissance', 20, order='asc')

Nous observons que plus la puissance est élevée plus les émissions de CO2 sont hautes.

### Analyse du CO2 en fonction de la classe de Cylindrée


In [ ]:
analyser_hist_co2_par_variable(df, 'Classe_Cylindree', 20, order='asc')

Plus la cylindrée **augmente**, plus les émissions de CO2 **augmentent**.

On passe d’environ **125 g/km** pour les petites cylindrées à plus de **270 g/km** pour les très grosses.

# <font color='#3585CD'>Corrélation des variables numériques</font>

In [ ]:
variables_selectionnees = ['Masse à vide', 'CO2', 'Cylindrée moteur', 'Puissance moteur', 'Consommation carburant']
plot_correlation_matrix(df[variables_selectionnees])

Nous observons : 
* une **très forte** corrélation entre la consommation de carburant et le CO2.
* une corrélation **modérée** entre la cylindrée et le CO2
* une corrélation **modérée** entre la puissance et le CO2
* une corrélation **faible** entre le poids et le CO2

# <font color='#3585CD'>Analyse des outliers</font>

## Masse à vide

In [ ]:
detecter_outliers(df['Masse à vide'])

Nous voyons la présence d’outliers de part et d’autre :

* Les plus légers : probablement des citadines
* Les plus lourds : SUV, hybrides rechargeables, utilitaires

Nous pouvons vérfier si ce sont seulement des valeurs **extrêmes** ou des valeurs **aberrantes**, auquel cas il faudra les traiter.

In [ ]:
df_outliers_masse = df[df['Masse à vide'] < 750].sort_values(by='Masse à vide', ascending=True)
df_outliers_masse.head(20)

In [ ]:
df_outliers_masse = df[df['Masse à vide'] > 2500].sort_values(by='Masse à vide', ascending=True)
df_outliers_masse.tail(20)

Après analyse, ces valeurs ne sont pas aberrantes. Elles sont juste extrêmes. Nous décidons de les garder car elles sont représentatives de certains véhicules.

## Cylindrée moteur

In [ ]:
detecter_outliers(df['Cylindrée moteur'])

Il y a beaucoup d'outliers à droite (> 2800 cm³), dont un à 8000 cm³. Ce sont sans doute des véhicules de sport ou luxe.

Un outlier est siuté à gauche.

Nous pouvons vérfier si ce sont seulement des valeurs **extrêmes** ou des valeurs **aberrantes**, auquel cas il faudra les traiter.

In [ ]:
df_outliers_CO2 = df[df['Cylindrée moteur'] > 4000].sort_values(by='Cylindrée moteur', ascending=True)
df_outliers_CO2.tail(20)

Après analyse, ces valeurs ne sont pas aberrantes. Elles sont juste extrêmes. Nous décidons de les garder car elles sont représentatives de certains véhicules.

## CO2

In [ ]:
detecter_outliers(df['CO2'])

La distribution est **asymétrique** à droite : il y a une longue traîne. Cela signifie que quelques véhicules émettent beaucoup plus de CO2 que la majorité.

Nous pouvons vérfier si ce sont seulement des valeurs **extrêmes** ou des valeurs **aberrantes**, auquel cas il faudra les traiter.


In [ ]:
df_outliers_CO2 = df[df['CO2'] > 400].sort_values(by='CO2', ascending=True)
df_outliers_CO2.tail(20)

Après analyse, ces valeurs ne sont pas aberrantes. Elles sont juste extrêmes. Nous décidons de les garder car elles sont représentatives de certains véhicules.

## Puissance moteur

In [ ]:
detecter_outliers(df['Puissance moteur'])

La **majorité** des véhicules ont entre 90 et 180 chevaux.

Beaucoup d’outliers sont **observés à droite**, représentant sans doute les véhicules puissants (sportives, gros SUV, ...)

2 véhicules sont particulièrement **extrêmes** (> 1000 ch).

Nous pouvons vérfier si ce sont seulement des valeurs **extrêmes** ou des valeurs **aberrantes**, auquel cas il faudra les traiter.


In [ ]:
df_outliers_Consommation = df[df['Puissance moteur'] > 15].sort_values(by='Puissance moteur', ascending=True)
df_outliers_Consommation.tail(20)

Après analyse, ces valeurs ne sont pas aberrantes. Elles sont juste extrêmes. Nous décidons de les garder car elles sont représentatives de certains véhicules.

## Consommation carburant

In [ ]:
detecter_outliers(df['Consommation carburant'])

 Nous obervons une **longue traîne à droite** : ces véhicules sont très énergivores.

Les extrêmes > 15 L/100 km sont très rares mais à surveiller.

Nous pouvons vérfier si ce sont seulement des valeurs **extrêmes** ou des valeurs **aberrantes**, auquel cas il faudra les traiter.


In [ ]:
df_outliers_Consommation = df[df['Consommation carburant'] > 15].sort_values(by='Consommation carburant', ascending=True)
df_outliers_Consommation.tail(20)

Après analyse, ces valeurs ne sont pas aberrantes. Elles sont juste extrêmes. Nous décidons de les garder car elles sont représentatives de certains véhicules.

# <font color='#3585CD'>Visualisation globale graphique</font>

In [ ]:
#numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
numeric_cols = ['Masse à vide', 'CO2', 'Cylindrée moteur', 'Puissance moteur', 'Consommation carburant']

hue_var = 'Carburant'
if visualisation == 'plotly': 
    fig = px.scatter_matrix(df, dimensions=df.select_dtypes(include=['number']).columns,
                            color='Carburant', title="Pairplot des variables numériques")

    fig.update_layout(height=900, width=1200)
    fig.show()
elif visualisation == 'seaborn':
    # Pairplot Seaborn
    sns.pairplot(df[numeric_cols + [hue_var]], hue=hue_var, corner=True)
    plt.suptitle("Pairplot des variables numériques", y=1.02)
    plt.show()

In [ ]:
plot_correlation_matrix(df[numeric_cols])

# <font color='#3585CD'>Distribution de la variable cible</font>

In [ ]:
analyser_variables_numeriques(df, ['CO2'])

In [ ]:
# Histogramme en fonction du type de motorisation
if visualisation == 'plotly':
    fig = px.histogram(df, x="CO2", color="Carburant", nbins=50, barmode="overlay",
                    title="Distribution des émissions de CO2 par type de carburant")
    fig.show()
elif visualisation == 'seaborn':
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x="CO2", hue="Carburant", bins=50, stat="count", element="step", common_norm=False)

    plt.title("Distribution des émissions de CO2 par type de carburant")
    plt.xlabel("Émissions de CO2 (g/km)")
    plt.ylabel("Nombre de véhicules")
    plt.tight_layout()
    plt.show()

# <font color='#3585CD'>Quelles variables garder pour prédire le CO2 ?</font>

Dans un premier temps, nous garderons les variables ci-dessous pour commencer la modélisation :
* **Masse à vide** : un véhicule plus lourd a souvent des émissions plus élevées.
* **Cylindrée moteur** : une plus grande cylindrée est souvent associée à une plus grande consommation et donc plus d’émissions.
* **Puissance moteur** : un moteur plus puissant a tendance à consommer plus de carburant.
* **Carburant** : Essence, diesel, hybride… Chaque type influence les émissions. La variable Carburant sera encodée.

* **Consommation carburant** : directement liée aux émissions de CO2. Cependant sa forte corrélation peut expliquer à elle seule les émissions de CO2. Nous nous questionnerons au moment de la modélisation si nous la gardons ou pas.

Nous **excluerons** les variables **Marque** et **Modèle** qui ne sont pas directement liées aux émissions de CO2.

Pourquoi exclure les variables **Marque** et **Modèle** ?
* Ce sont des variables catégoriques à très haute **cardinalité**
* L'influence sur le CO₂ passe par des **caractéristiques techniques**

Après la mise en place des premiers modèles, nous reviendrons éventuellement sur notre dataset pour analyser la combinaison sur d'autres variables.

# <font color='#3585CD'>Export du dataset final</font>

Nous vérifions une dernière fois si des doublons sont présents :

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df

In [ ]:
df_final = df.drop(['Classe_Cylindree', 'Classe_Puissance', 'ICSM', 'ICE', 'IDEC', 'ICS'], axis=1)
df_final

In [ ]:
print(f"Notre dataset final est composé de {df_final.shape[0]} lignes")

In [ ]:
# Vérifier si on est sur Google Colab
import os
try:
    import google.colab
    ON_COLAB = True
    dataset_path = "/content/drive/My Drive/Formation DS/Projet CO2/NOV24-CDS-CO2/notebooks/datasets/Dataset_final/datas_nettoyees_model_FR.csv"

    # Monter Google Drive si ce n'est pas déjà fait
    from google.colab import drive
    drive.mount('/content/drive')

except ImportError:
    ON_COLAB = False
    dataset_path = "datasets/Dataset_final/datas_nettoyees_model_FR.csv"

# Sauvegarde du DataFrame
df_final.to_csv(dataset_path, index=False)

# Vérification de l'enregistrement
if os.path.exists(dataset_path):
    print(f"Le fichier a bien été enregistré à l'emplacement : {dataset_path}")
else:
    print("Problème lors de l'enregistrement du fichier.")
